# FID004 · 日内 v3 完整研究

本 Notebook 运行分别注册的日盘与夜盘开盘区间事件。它核对确切的冻结配置，写出每个品种的 PnL 与完整交易，并检查结构保护位和假突破行为。

In [1]:
import pandas as pd
from infra.config import PROJECT, factor_strategies, product_ids
from infra.runner import DEFAULT_END, DEFAULT_START, DEFAULT_WARMUP, run_factor

FACTOR_ID = 'FID004'
RUN_DIR = PROJECT / FACTOR_ID.lower() / 'runs' / 'v3_10y'
PRODUCTS = product_ids()

## 1. 冻结的日盘/夜盘注册

In [ ]:
pd.DataFrame(factor_strategies(FACTOR_ID)).T

## 2. 完整十年运行

In [ ]:
summary = run_factor(FACTOR_ID, PRODUCTS, warmup_start=DEFAULT_WARMUP, start=DEFAULT_START, end_exclusive=DEFAULT_END)
summary

## 3. 日盘/夜盘与品种 PnL

In [ ]:
strategy_id = factor_strategies(FACTOR_ID)[0]['strategy_id']
product_id = 'SHFE.RB'
pnl = pd.read_csv(RUN_DIR / strategy_id / product_id / 'pnl.csv', parse_dates=['date'])
pnl.set_index('date')['cumulative_net_pnl'].plot(figsize=(13, 4), title=f'{strategy_id} · {product_id}')

## 4. 结构性退出与假突破

In [ ]:
trades = pd.read_csv(RUN_DIR / strategy_id / product_id / 'trades.csv')
display(trades.head())
display(trades.groupby('exit_reason').agg(trades=('trade_id', 'count'), net_pnl=('net_pnl', 'sum')))
false_break = trades['exit_reason'].isin(['signal_zero', 'structural_stop', 'structural_stop_gap']).mean()
print(f'假突破占比：{false_break:.1%}')